In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [5]:
import time
import json
from typing import cast
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
import keras
from keras import layers
from visualize import save_confusion_matrix, save_class_metrics, save_class_distribution

In [6]:
BASE_DIR = Path.cwd()
DATA_DIR    = BASE_DIR / "data"
TRAIN_DIR   = DATA_DIR / "train"
VAL_DIR     = DATA_DIR / "val"
TEST_DIR    = DATA_DIR / "test"
MODEL_PATH  = BASE_DIR / "weather_model.h5"
HISTORY_PNG = BASE_DIR / "training_history.png"

In [7]:
IMG_SIZE    = (150, 150)
BATCH_SIZE  = 32
EPOCHS      = 50      
NUM_CLASSES = 5
SEED        = 42

In [8]:
CLASSES = ["cloudy", "foggy", "rainy", "shine", "sunrise"]

In [9]:
_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.10),            # +-36 degrees
        layers.RandomZoom((-0.20, 0.20)),        # zoom in or out up to 20%
        layers.RandomTranslation(height_factor=0.10, width_factor=0.10),
        layers.RandomBrightness(factor=0.15, value_range=(0, 1)),
        layers.RandomContrast(factor=0.15),
    ],
    name="augmentation",
)

In [10]:
def _normalize(images, labels):

    return tf.cast(images, tf.float32) / 255.0, labels

In [11]:
def _augment(images, labels):

    return _augmentation(images, training=True), labels

In [12]:
def load_dataset(directory: Path, *, shuffle: bool, augment: bool) -> tf.data.Dataset:

    ds = cast(tf.data.Dataset, keras.utils.image_dataset_from_directory(
        str(directory),
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="int",
        class_names=CLASSES,
        shuffle=shuffle,
        seed=SEED,
    ))
    ds = ds.map(_normalize, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(_augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

In [13]:
def compute_weights() -> dict:

    counts = np.array(
        [len(list((TRAIN_DIR / cls).iterdir())) for cls in CLASSES],
        dtype=int,
    )
    y_flat  = np.repeat(np.arange(NUM_CLASSES), counts)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(NUM_CLASSES),
        y=y_flat,
    )
    weight_dict = {i: float(w) for i, w in enumerate(weights)}
    print("\nClass weights (imbalance correction):")
    for i, cls in enumerate(CLASSES):
        print(f"  [{i}] {cls:<10}  weight={weight_dict[i]:.4f}  ({counts[i]} samples)")
    return weight_dict

In [14]:
def build_model() -> keras.Model:

    reg = keras.regularizers.l2(1e-4)

    inputs = keras.Input(shape=(*IMG_SIZE, 3), name="input")


    x = layers.Conv2D(32, (3, 3), padding="same", use_bias=False, name="conv1")(inputs)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.Activation("relu", name="relu1")(x)
    x = layers.MaxPooling2D((2, 2), name="pool1")(x)

    x = layers.Conv2D(64, (3, 3), padding="same", use_bias=False, name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.Activation("relu", name="relu2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool2")(x)

    x = layers.Conv2D(128, (3, 3), padding="same", use_bias=False, name="conv3")(x)
    x = layers.BatchNormalization(name="bn3")(x)
    x = layers.Activation("relu", name="relu3")(x)
    x = layers.MaxPooling2D((2, 2), name="pool3")(x)

    x = layers.Conv2D(256, (3, 3), padding="same", use_bias=False, name="conv4")(x)
    x = layers.BatchNormalization(name="bn4")(x)
    x = layers.Activation("relu", name="relu4")(x)
    x = layers.MaxPooling2D((2, 2), name="pool4")(x)

 
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=reg, name="fc1")(x)
    x = layers.Dropout(0.40, name="dropout")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="output")(x)

    model = keras.Model(inputs, outputs, name="WeatherCNN_v2")
    return model

In [15]:
def save_history_plot(history: keras.callbacks.History) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history["accuracy"],     label="Train")
    axes[0].plot(history.history["val_accuracy"], label="Validation")
    axes[0].set_title("Accuracy over Epochs")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history.history["loss"],     label="Train")
    axes[1].plot(history.history["val_loss"], label="Validation")
    axes[1].set_title("Loss over Epochs")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(str(HISTORY_PNG), dpi=150)
    plt.close(fig)
    print(f"Training history plot saved -> {HISTORY_PNG}")

In [17]:
if __name__ == "__main__":


    for d, name in [(TRAIN_DIR, "train"), (VAL_DIR, "val"), (TEST_DIR, "test")]:
        if not d.exists():
            raise FileNotFoundError(
                f"[ERROR] '{name}' directory not found: {d}\n"
                "Run prepare_dataset.py first."
            )


    print("Loading datasets ...")
    train_ds = load_dataset(TRAIN_DIR, shuffle=True,  augment=True)
    val_ds   = load_dataset(VAL_DIR,   shuffle=False, augment=False)
    test_ds  = load_dataset(TEST_DIR,  shuffle=False, augment=False)

    class_weights = compute_weights()


    print("\nBuilding model ...")
    model = build_model()
    model.summary()

    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),

        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1,
        ),

        keras.callbacks.ModelCheckpoint(
            str(MODEL_PATH),
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
        ),
    ]


    print(f"\nStarting training (max {EPOCHS} epochs, early stopping enabled) ...\n")
    t_start = time.time()
    try:
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            callbacks=callbacks,
            class_weight=class_weights,
        )
    except Exception as e:
        print(f"Error during training: {e}")
        print("Possible cause: Missing or corrupted files in the dataset. Run prepare_dataset.py again to regenerate the dataset.")
        raise
    training_time = time.time() - t_start

    save_history_plot(history)

    
    print("\n" + "=" * 50)
    print("FINAL EVALUATION ON HELD-OUT TEST SET")
    print("=" * 50)
    test_loss, test_acc = model.evaluate(test_ds, verbose="auto")
    print(f"\n  Test Loss     : {test_loss:.4f}")
    print(f"  Test Accuracy : {test_acc * 100:.2f}%")
    print("=" * 50)
    print(f"\nBest model saved -> {MODEL_PATH}")

    trainable_params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
    scratch_metrics = {
        "best_train_accuracy": float(max(history.history["accuracy"])),
        "best_val_accuracy":   float(max(history.history["val_accuracy"])),
        "test_accuracy":       float(test_acc),
        "test_loss":           float(test_loss),
        "training_time_s":     float(training_time),
        "trainable_params":    trainable_params,
    }
    with open(BASE_DIR / "scratch_metrics.json", "w") as f:
        json.dump(scratch_metrics, f, indent=2)
    print(f"Metrics saved -> {BASE_DIR / 'scratch_metrics.json'}")

    save_class_distribution(TRAIN_DIR, VAL_DIR, TEST_DIR,
                            BASE_DIR / "class_distribution.png")
    y_true, y_pred = save_confusion_matrix(
        model, test_ds,
        BASE_DIR / "confusion_matrix.png",
        title="Confusion Matrix - Scratch CNN (Test Set)",
    )
    save_class_metrics(
        y_true, y_pred,
        BASE_DIR / "class_metrics.png",
        title="Per-class Metrics - Scratch CNN (Test Set)",
    )

Loading datasets ...
Found 1048 files belonging to 5 classes.


Found 226 files belonging to 5 classes.
Found 226 files belonging to 5 classes.

Class weights (imbalance correction):
  [0] cloudy      weight=0.9981  (210 samples)
  [1] foggy       weight=0.9981  (210 samples)
  [2] rainy       weight=0.9981  (210 samples)
  [3] shine       weight=1.2046  (174 samples)
  [4] sunrise     weight=0.8590  (244 samples)

Building model ...


Model: "WeatherCNN_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 150, 150, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 150, 150, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1 (Activation)              │ (None, 150, 150, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 75, 75, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 75, 75, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2 (Activation)              │ (None, 75, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 37, 37, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 37, 37, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu3 (Activation)              │ (None, 37, 37, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling2D)            │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 18, 18, 256)    │       294,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn4 (BatchNormalization)        │ (None, 18, 18, 256)    │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu4 (Activation)              │ (None, 18, 18, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool4 (MaxPooling2D)            │ (None, 9, 9, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 456,933 (1.74 MB)

...


Epoch 14: finished saving model to c:\Users\h\Documents\Weatherimg\weather_model.h5
33/33 ━━━━━━━━━━━━━━━━━━━━ 25s 740ms/step - accuracy: 0.8435 - loss: 0.4396 - val_accuracy: 0.7124 - val_loss: 0.7536 - learning_rate: 2.5000e-04
Epoch 15/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 710ms/step - accuracy: 0.8687 - loss: 0.3999
Epoch 15: val_accuracy did not improve from 0.71239
33/33 ━━━━━━━━━━━━━━━━━━━━ 25s 744ms/step - accuracy: 0.8664 - loss: 0.4026 - val_accuracy: 0.6372 - val_loss: 1.0776 - learning_rate: 2.5000e-04
Epoch 16/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 710ms/step - accuracy: 0.8836 - loss: 0.3647
Epoch 16: val_accuracy improved from 0.71239 to 0.86726, saving model to c:\Users\h\Documents\Weatherimg\weather_model.h5



Epoch 16: finished saving model to c:\Users\h\Documents\Weatherimg\weather_model.h5
33/33 ━━━━━━━━━━━━━━━━━━━━ 25s 745ms/step - accuracy: 0.8769 - loss: 0.3740 - val_accuracy: 0.8673 - val_loss: 0.4252 - learning_rate: 2.5000e-04
Epoch 17/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 685ms/step - accuracy: 0.8775 - loss: 0.3761
Epoch 17: val_accuracy did not improve from 0.86726
33/33 ━━━━━━━━━━━━━━━━━━━━ 25s 722ms/step - accuracy: 0.8769 - loss: 0.3698 - val_accuracy: 0.8142 - val_loss: 0.4262 - learning_rate: 2.5000e-04
Epoch 18/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 674ms/step - accuracy: 0.8736 - loss: 0.3861
Epoch 18: val_accuracy improved from 0.86726 to 0.88938, saving model to c:\Users\h\Documents\Weatherimg\weather_model.h5



Epoch 18: finished saving model to c:\Users\h\Documents\Weatherimg\weather_model.h5
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 709ms/step - accuracy: 0.8769 - loss: 0.3858 - val_accuracy: 0.8894 - val_loss: 0.3296 - learning_rate: 2.5000e-04
Epoch 19/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 727ms/step - accuracy: 0.8572 - loss: 0.3848
Epoch 19: val_accuracy improved from 0.88938 to 0.91150, saving model to c:\Users\h\Documents\Weatherimg\weather_model.h5



Epoch 19: finished saving model to c:\Users\h\Documents\Weatherimg\weather_model.h5
33/33 ━━━━━━━━━━━━━━━━━━━━ 26s 764ms/step - accuracy: 0.8731 - loss: 0.3614 - val_accuracy: 0.9115 - val_loss: 0.3110 - learning_rate: 2.5000e-04
Epoch 20/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 657ms/step - accuracy: 0.8701 - loss: 0.3735
Epoch 20: val_accuracy did not improve from 0.91150
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 690ms/step - accuracy: 0.8788 - loss: 0.3661 - val_accuracy: 0.7788 - val_loss: 0.5185 - learning_rate: 2.5000e-04
Epoch 21/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 659ms/step - accuracy: 0.8770 - loss: 0.3426
Epoch 21: val_accuracy did not improve from 0.91150
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 695ms/step - accuracy: 0.8845 - loss: 0.3450 - val_accuracy: 0.8407 - val_loss: 0.4607 - learning_rate: 2.5000e-04
Epoch 22/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 677ms/step - accuracy: 0.8544 - loss: 0.4101
Epoch 22: val_accuracy improved from 0.91150 to 0.91593, saving model to c:\Users\h\Documents\Weatherimg\weather


Epoch 22: finished saving model to c:\Users\h\Documents\Weatherimg\weather_model.h5
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 711ms/step - accuracy: 0.8635 - loss: 0.3939 - val_accuracy: 0.9159 - val_loss: 0.3227 - learning_rate: 2.5000e-04
Epoch 23/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 645ms/step - accuracy: 0.8809 - loss: 0.3816
Epoch 23: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 23: val_accuracy did not improve from 0.91593
33/33 ━━━━━━━━━━━━━━━━━━━━ 23s 677ms/step - accuracy: 0.8826 - loss: 0.3714 - val_accuracy: 0.8540 - val_loss: 0.3336 - learning_rate: 2.5000e-04
Epoch 24/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 685ms/step - accuracy: 0.8874 - loss: 0.3202
Epoch 24: val_accuracy did not improve from 0.91593
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 719ms/step - accuracy: 0.8922 - loss: 0.3254 - val_accuracy: 0.8717 - val_loss: 0.3845 - learning_rate: 1.2500e-04
Epoch 25/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 675ms/step - accuracy: 0.9044 - loss: 0.2841
Epoch 25: val_accuracy did not imp